# Regular Expressions (Regex) in Python

## Learning Objectives

By the end of this lesson, you should be able to:

- Explain what Regular Expressions are and why they are useful.
- Understand why raw strings are commonly used for Regex patterns.
- Build patterns using literal characters, metacharacters, character classes, quantifiers, and anchors.
- Use common Regex functions to search, extract, validate, replace, and split text.
- Use groups to extract related pieces of information.
- Recognise advanced features such as greedy matching, lookarounds, and backreferences.
- Apply Regex to clean and extract information from text columns in Pandas.

---

# 1. Introduction to Regular Expressions

## 1.1 What Are Regular Expressions?

Imagine you receive hundreds of customer messages. Some contain order numbers, dates, phone numbers, or email addresses.

You need to extract every order number, but you do not know the actual numbers in advance.

Instead of searching for one exact value, you need a way to describe what an order number looks like.

This is what **Regular Expressions (Regex)** allow us to do.

A Regular Expression is a pattern used to search, match, extract, validate, replace, or split text.

For example, if an order reference always starts with `ORD-` followed by four digits, we can describe it with this pattern:

```python
r"ORD-\d{4}"
```

It can match `ORD-1024`, `ORD-5831`, and `ORD-9002` without knowing those values beforehand.

## 1.2 Why Do We Need Regex?

Python string methods are useful when we know the exact text we want to find.

```python
message = "Your delivery is scheduled for tomorrow."

print("delivery" in message)
```

Output:

```text
True
```

But what if we have many messages and want to extract every order reference?

```python
messages = """
Order ORD-1024 has been dispatched.
Order ORD-5831 is being prepared.
Order ORD-9002 has been delivered.
"""

```

We do not want to search for each order number individually. We want to describe the common structure: `ORD-` followed by four digits.

```python
import re

order_ids = re.findall(r"ORD-\d{4}", messages)

print(order_ids)
```

Output:

```text
['ORD-1024', 'ORD-5831', 'ORD-9002']
```

**The key idea:** string methods often look for known text; Regex lets us search for text that follows a pattern.

## 1.3 Where Is Regex Used?

| Application | Example |
|---|---|
| Data extraction | Extracting order IDs, email addresses, and dates |
| Data cleaning | Removing unwanted symbols from product codes |
| Data validation | Checking whether an ID follows a required format |
| Text analysis | Finding words or phrases in customer feedback |
| Data preparation | Separating useful information from unstructured text |

Regex is also used in SQL databases, text editors, JavaScript, Java, and command-line tools.

---

# 2. Getting Started with Regex in Python

## 2.1 Importing the Regex Module

Python provides a built-in module called `re`.

```python
import re
```

We will use `re.findall()` initially to test the patterns we create. We will explore other Regex functions in Section 7.

## 2.2 Understanding Raw Strings

You will often see the letter `r` before the quotation marks in a Regex pattern:

```python
pattern = r"\d+"
```

This is called a **raw string**.

### Why do we use it?

In normal Python strings, the backslash (`\`) can introduce special characters.

For example:

```python
print("First line\nSecond line")
```

Output:

```text
First line
Second line
```

Python interprets `\n` as a new line.

Regex also uses backslashes to give patterns special meanings:

- `\d` means a digit.
- `\w` means a word character.
- `\s` means whitespace, such as a space or tab.

If Python interprets a backslash combination before Regex receives it, the pattern may not mean what we intended.

### The solution: use a raw string

```python
pattern = r"\d+"
```

The `r` tells Python to preserve backslashes instead of interpreting most backslash combinations as Python escape sequences. Regex then receives `\d+` and understands it as one or more digits.

### Normal strings vs raw strings

| Python string | What Python does |
|---|---|
| `"\n"` | Interprets `\n` as a new line |
| `r"\n"` | Preserves the backslash and `n` |
| `"\\d+"` | Represents the Regex pattern `\d+` |
| `r"\d+"` | Also represents the Regex pattern `\d+` |

Both `"\\d+"` and `r"\d+"` can represent the same Regex pattern. Raw strings are simply easier to read and write.

### See it in action

```python
import re

message = "Your verification code is 482916."

pattern = r"\d+"

result = re.findall(pattern, message)

print(result)
```

Output:

```text
['482916']
```

What happened?

- `r` tells Python to preserve the backslash in the pattern.
- `\d+` tells Regex to find one or more digits together.
- `re.findall()` returns all matching portions in a list.

**Remember:** `r` is for Python; `\d`, `\w`, and `\s` are instructions for the Regex engine.

For this lesson, use raw strings whenever you write Regex patterns.

## 2.3 Our First Regex Pattern

A Regex pattern can also be ordinary text. It does not have to contain special symbols.

```python
message = "Payment received. Your order is ready."

result = re.findall(r"Payment", message)

print(result)
```

Output:

```text
['Payment']
```

The pattern `Payment` looks for that exact sequence of characters.

For now, think of `re.findall()` as a tool that finds every occurrence of the pattern and returns the matches in a list.

---

# 3. Understanding Regex Patterns

A Regex pattern is made up of characters and symbols that describe what we want to find.

Two important ideas:

1. **Literal characters** match the exact characters written in the pattern.
2. **Metacharacters** have special meanings in Regex.

## 3.1 Literal Characters

Literal characters match themselves.

```python
message = "The parcel is ready. The parcel will be collected today."

matches = re.findall(r"parcel", message)

print(matches)
```

Output:

```text
['parcel', 'parcel']
```

The pattern `parcel` finds both occurrences of that exact sequence.

### Student Challenge 1: Literal Patterns

Given:

```python
message = "The laptop is ready. The monitor is ready. The laptop has been tested."
```

Use `re.findall()` to:

1. Find every occurrence of `laptop`.
2. Find every occurrence of `monitor`.
3. Search for `keyboard`.

### Reference Solution

```python
import re

message = "The laptop is ready. The monitor is ready. The laptop has been tested."

laptops = re.findall(r"laptop", message)
monitors = re.findall(r"monitor", message)
keyboards = re.findall(r"keyboard", message)

print("Laptops:", laptops)
print("Monitors:", monitors)
print("Keyboards:", keyboards)
```

Output:

```text
Laptops: ['laptop', 'laptop']
Monitors: ['monitor']
Keyboards: []
```

If a pattern is not found, `findall()` returns an empty list.

## 3.2 Metacharacters

Metacharacters are special Regex symbols. They help us describe patterns instead of exact words.

### Common metacharacters

| Metacharacter | Meaning | Example | What it matches |
|---|---|---|---|
| `.` | Any one character except a newline by default | `p.n` | `pan`, `pen`, `pin` |
| `[]` | Any one character inside the brackets | `[ABC]` | `A`, `B`, or `C` |
| `[^...]` | Any one character not listed | `[^0-9]` | A character that is not a digit |
| `()` | Groups parts of a pattern | `(ab)` | The sequence `ab` as a group |
| `\|` | OR (alternation) | `tea\|coffee` | `tea` or `coffee` |
| `\` | Escapes a special character or introduces a special sequence | `\.` | A literal full stop |

> In a Regex pattern, the vertical bar is written as `|`. It means OR.

### The dot (`.`)

The dot matches any single character except a newline by default.

```python
text = "pen, pan, pin, pot, ten"

matches = re.findall(r"p.n", text)

print(matches)
```

Output:

```text
['pen', 'pan', 'pin']
```

Break down `p.n`:

- `p` requires the letter `p`.
- `.` allows any one character.
- `n` requires the letter `n`.

It matches `pen`, `pan`, and `pin`, but not `pot` or `ten`.

### Escaping special characters

Some characters have special meanings in Regex. If we want to find the actual character, we place a backslash before it.

For example, a full stop (`.`) normally means any character. To find an actual full stop:

```python
text = "The report is ready. The meeting starts soon."

matches = re.findall(r"\.", text)

print(matches)
```

Output:

```text
['.', '.']
```

The backslash tells Regex to treat the dot as an ordinary full-stop character.

The same idea applies to other special characters:

```python
prices = "KES 500, KES 1,200, KES 750"

matches = re.findall(r",", prices)

print(matches)
```

Here, the comma is not a Regex metacharacter, so it can be matched directly.

### Student Challenge 2: Working with Metacharacters

Given:

```python
text = "The codes are pen, pan, pin and pot. The file is ready."
```

Use `re.findall()` to:

1. Find words that start with `p`, have any one character in the middle, and end with `n`.
2. Find the full stop.
3. Find the word `file`.

### Reference Solution

```python
import re

text = "The codes are pen, pan, pin and pot. The file is ready."

p_words = re.findall(r"p.n", text)
full_stops = re.findall(r"\.", text)
file_word = re.findall(r"file", text)

print("Matching words:", p_words)
print("Full stops:", full_stops)
print("File:", file_word)
```

Output:

```text
Matching words: ['pen', 'pan', 'pin']
Full stops: ['.', '.']
File: ['file']
```

---

# 4. Character Classes

The dot matches any one character. But what if we only want to allow certain characters?

For example, suppose a product code starts with `P`, ends with `7`, and has either `A` or `B` in the middle.

We can use a **character class**.

Character classes are written inside square brackets: `[]`.

## 4.1 Matching Specific Characters

```python
codes = "PA7, PB7, PC7, PD7"

matches = re.findall(r"P[AB]7", codes)

print(matches)
```

Output:

```text
['PA7', 'PB7']
```

Break down `P[AB]7`:

- `P` matches the letter `P`.
- `[AB]` matches either `A` or `B`.
- `7` matches the digit `7`.

Only `PA7` and `PB7` match.

## 4.2 Character Ranges

Instead of listing every character, we can specify a range.

| Pattern | Meaning |
|---|---|
| `[a-z]` | Any lowercase English letter |
| `[A-Z]` | Any uppercase English letter |
| `[0-9]` | Any digit |
| `[a-zA-Z]` | Any uppercase or lowercase English letter |
| `[a-zA-Z0-9]` | Any English letter or digit |

Example:

```python
product_code = "ABX204"

letters = re.findall(r"[A-Z]", product_code)
digits = re.findall(r"[0-9]", product_code)

print("Letters:", letters)
print("Digits:", digits)
```

Output:

```text
Letters: ['A', 'B', 'X']
Digits: ['2', '0', '4']
```

Each character class match is one character. We will learn how to match groups of characters using quantifiers in the next section.

## 4.3 Negated Character Classes (`[^...]`)

Sometimes we want to match everything except certain characters.

We use `^` at the beginning of a character class to mean NOT.

```python
reference = "INV45A"

matches = re.findall(r"[^0-9]", reference)

print(matches)
```

Output:

```text
['I', 'N', 'V', 'A']
```

The pattern `[^0-9]` matches any character that is not a digit.

Compare:

- `[0-9]` matches digits.
- `[^0-9]` matches characters that are not digits.

**Important:** the meaning of `^` depends on where it appears.

- Inside square brackets at the beginning, it means NOT.
- Outside square brackets, it represents the beginning of a string (covered in Section 6).

## 4.4 Common Shortcuts for Character Classes

Regex provides shortcuts for commonly used character classes.

| Pattern | Meaning |
|---|---|
| `\d` | Any digit |
| `\D` | Any non-digit |
| `\w` | Any word character: letters, digits, or underscore |
| `\W` | Any non-word character |
| `\s` | Any whitespace character, such as a space, tab, or newline |
| `\S` | Any non-whitespace character |

Let's use them:

```python
text = "Invoice INV204 was paid on 12 June."

# Find each digit
print(re.findall(r"\d", text))

# Find groups of word characters
print(re.findall(r"\w+", text))

# Find whitespace characters
print(re.findall(r"\s", text))
```

Output:

```text
['2', '0', '4', '1', '2']
['Invoice', 'INV204', 'was', 'paid', 'on', '12', 'June']
[' ', ' ', ' ', ' ', ' ', ' ', ' ']
```

Notice:

- `\d` matches digits one at a time.
- `\w+` matches groups of consecutive word characters.
- `\s` matches each whitespace character.

The uppercase shortcuts match the opposite character category of their lowercase versions.

### Student Challenge 3: Character Classes

Given:

```python
record = "Order ID: AB-2047 | Status: PAID"
```

Use Regex to:

1. Extract all digits.
2. Extract all uppercase letters.
3. Extract all lowercase letters.
4. Extract all characters that are not letters or digits.
5. Extract groups of word characters.

### Reference Solution

```python
import re

record = "Order ID: AB-2047 | Status: PAID"

digits = re.findall(r"\d", record)
uppercase = re.findall(r"[A-Z]", record)
lowercase = re.findall(r"[a-z]", record)
non_alphanumeric = re.findall(r"[^A-Za-z0-9]", record)
words = re.findall(r"\w+", record)

print("Digits:", digits)
print("Uppercase:", uppercase)
print("Lowercase:", lowercase)
print("Non-alphanumeric:", non_alphanumeric)
print("Words:", words)
```

Explanation: `[^A-Za-z0-9]` matches anything that is not an English letter or digit. This includes spaces, punctuation, and symbols.

---

# 5. Quantifiers

Character classes define which characters can match. **Quantifiers** define how many times a character or pattern can appear.

For example:

- A product code may contain several digits.
- A username may contain between 5 and 12 characters.
- A word may contain one or more letters.

## 5.1 Common Quantifiers

| Quantifier | Meaning | Example |
|---|---|---|
| `*` | Zero or more occurrences | `ab*` |
| `+` | One or more occurrences | `ab+` |
| `?` | Zero or one occurrence | `colou?r` |
| `{n}` | Exactly `n` occurrences | `\d{4}` |
| `{n,}` | At least `n` occurrences | `\d{2,}` |
| `{n,m}` | Between `n` and `m` occurrences | `\d{2,4}` |

## 5.2 Zero or More (`*`)

The `*` quantifier matches zero or more occurrences of the character or pattern immediately before it.

```python
codes = "A7 AB7 ABB7 ABBB7"

matches = re.findall(r"AB*7", codes)

print(matches)
```

Output:

```text
['A7', 'AB7', 'ABB7', 'ABBB7']
```

The pattern `AB*7` means:

- `A` must appear.
- `B*` means zero or more `B` characters.
- `7` must appear.

This is why `A7` also matches: it contains zero `B` characters.

## 5.3 One or More (`+`)

The `+` quantifier requires at least one occurrence.

```python
codes = "A7 AB7 ABB7 ABBB7"

matches = re.findall(r"AB+7", codes)

print(matches)
```

Output:

```text
['AB7', 'ABB7', 'ABBB7']
```

Unlike `*`, the `+` quantifier requires at least one `B`.

## 5.4 Zero or One (`?`)

The `?` quantifier makes the preceding character optional.

For example, both `organize` and `organise` are commonly used spellings. We can make the `z` optional:

```python
text = "Please organize the files. Please organise the folders."

matches = re.findall(r"organi[sz]e", text)

print(matches)
```

Output:

```text
['organize', 'organise']
```

Here, `[sz]` allows either `s` or `z`. To see the `?` quantifier itself, consider an optional hyphen in a reference:

```python
references = "REF1024 REF-2048 REF3096"

matches = re.findall(r"REF-?\d+", references)

print(matches)
```

Output:

```text
['REF1024', 'REF-2048', 'REF3096']
```

The `-?` means the hyphen may appear zero or one time.

## 5.5 Exact and Range-Based Repetition

Curly brackets `{}` let us specify the number of repetitions.

```python
codes = "A7 B42 C305 D2026 E12345"

# Exactly four digits
print(re.findall(r"\d{4}", codes))
```

Output:

```text
['2026']
```

More examples:

```python
codes = "A7 B42 C305 D2026 E12345"

# At least two digits
print(re.findall(r"\d{2,}", codes))

# Between two and four digits
print(re.findall(r"\d{2,4}", codes))
```

Output:

```text
['42', '305', '2026', '12345']
['42', '305', '2026', '1234']
```

The second pattern finds matching portions between two and four digits long. It does not automatically require the entire number to contain no additional digits.

### Student Challenge 4: Quantifiers

Given:

```python
codes = "X5 X55 X555 X5555"
```

Write patterns to:

1. Find one or more `5` characters after `X`.
2. Find exactly two `5` characters after `X`.
3. Find between two and four `5` characters after `X`.
4. Find codes where the hyphen is optional: `INV-204` and `INV204`.

### Reference Solution

```python
import re

codes = "X5 X55 X555 X5555"

one_or_more = re.findall(r"X5+", codes)
exactly_two = re.findall(r"X5{2}", codes)
between_two_and_four = re.findall(r"X5{2,4}", codes)

references = "INV-204 INV204"
optional_hyphen = re.findall(r"INV-?\d+", references)

print("One or more:", one_or_more)
print("Exactly two:", exactly_two)
print("Between two and four:", between_two_and_four)
print("Optional hyphen:", optional_hyphen)
```

**Remember:** quantifiers apply to the character or group immediately before them. `findall()` searches for matching portions of text, so a pattern may match part of a longer sequence unless you also define the boundaries.

---

# 6. Anchors and Word Boundaries

Sometimes we do not just want to find a pattern anywhere in a string. We may want to specify where it should appear.

Examples:

- A reference must start with `INV`.
- A filename must end with `.csv`.
- A word must appear as a complete word, not as part of another word.

Regex uses **anchors** and **word boundaries** to describe positions.

## 6.1 Beginning of a String (`^`)

The `^` symbol matches the beginning of a string.

```python
filename = "sales_report.csv"

result = re.findall(r"^sales", filename)

print(result)
```

Output:

```text
['sales']
```

The pattern only matches `sales` if it appears at the beginning.

## 6.2 End of a String (`$`)

The `$` symbol matches the end of a string.

```python
filename = "sales_report.csv"

result = re.findall(r"\.csv$", filename)

print(result)
```

Output:

```text
['.csv']
```

The pattern checks that `.csv` appears at the end. The dot is escaped because we want an actual full stop, not the any-character metacharacter.

## 6.3 Word Boundary (`\b`)

A word boundary marks the position between a word character and a non-word character, or the beginning or end of a string.

It is useful when we want a complete word rather than the same letters inside a longer word.

```python
text = "The app is useful. This application is popular."

print(re.findall(r"\bapp\b", text))
```

Output:

```text
['app']
```

The pattern `\bapp\b` matches `app` as a complete word. It does not match the beginning of `application`.

## 6.4 Difference Between a Word and a Word Boundary

```python
text = "The app is useful. This application is popular."

print(re.findall(r"app", text))
print(re.findall(r"\bapp\b", text))
```

Output:

```text
['app', 'app']
['app']
```

The first pattern also finds `app` inside `application`. The second only finds the complete word.

### Student Challenge 5: Anchors and Boundaries

Given:

```python
files = "sales.csv, sales.xlsx, report.csv"
```

Write patterns to:

1. Find `sales` only if it appears at the beginning of the string.
2. Find `.csv` only if it appears at the end of the string.
3. Find the complete word `sales` (not part of a longer word).

### Reference Solution

```python
import re

files = "sales.csv, sales.xlsx, report.csv"

beginning = re.findall(r"^sales", files)
ending = re.findall(r"\.csv$", files)
complete_word = re.findall(r"\bsales\b", files)

print("Beginning:", beginning)
print("Ending:", ending)
print("Complete word:", complete_word)
```

---

# 7. Common Regex Functions in Python

Now that we understand how to construct patterns, let's explore the functions Python provides for working with them.

## 7.1 Common Regex Functions

| Function | Purpose | Returns |
|---|---|---|
| `re.findall()` | Finds all matches | A list |
| `re.search()` | Finds the first match anywhere in the text | A Match Object or `None` |
| `re.match()` | Checks for a match at the beginning | A Match Object or `None` |
| `re.fullmatch()` | Checks whether the entire string matches | A Match Object or `None` |
| `re.finditer()` | Finds all matches one at a time | An iterator of Match Objects |
| `re.sub()` | Replaces matching text | A new string |
| `re.split()` | Splits text wherever the pattern matches | A list |
| `re.compile()` | Creates a reusable Regex pattern | A compiled pattern object |

## 7.2 `re.findall()`

`re.findall()` finds all occurrences of a pattern and returns them in a list.

```python
message = "Order ORD-1024 shipped. Order ORD-2048 is pending."

order_ids = re.findall(r"ORD-\d{4}", message)

print(order_ids)
```

Output:

```text
['ORD-1024', 'ORD-2048']
```

## 7.3 `re.search()`

`re.search()` looks for the first occurrence of a pattern anywhere in the text.

It returns a Match Object if a match is found. If no match is found, it returns `None`.

```python
message = "Payment reference: TXN-84721. Status: complete."

result = re.search(r"TXN-\d+", message)

if result:
    print("Transaction ID:", result.group())
else:
    print("No transaction ID found.")
```

Output:

```text
Transaction ID: TXN-84721
```

### Understanding the Match Object

The `.group()` method retrieves the matched text.

```python
message = "The parcel weighs 4.5 kg."

result = re.search(r"\d+\.\d+", message)

if result:
    print(result.group())
```

Output:

```text
4.5
```

Remember:

- `re.search()` finds the first match.
- `.group()` retrieves the matched text.
- Check that a match exists before calling `.group()`.

## 7.4 `re.match()`

`re.match()` checks for a pattern only at the beginning of the string.

```python
reference = "INV-2048"

result = re.match(r"INV", reference)

if result:
    print("Reference starts with INV")
else:
    print("Reference does not start with INV")
```

Output:

```text
Reference starts with INV
```

Now consider:

```python
reference = "Payment INV-2048"

result = re.match(r"INV", reference)

print(result)
```

Output:

```text
None
```

Although `INV` exists in the text, it is not at the beginning.

## 7.5 `re.fullmatch()`

`re.fullmatch()` checks whether the entire string follows the specified pattern.

It is useful when checking whether an input follows a required format.

Suppose a product code must contain exactly two uppercase letters followed by four digits:

```python
product_code = "AB2048"

result = re.fullmatch(r"[A-Z]{2}\d{4}", product_code)

if result:
    print("Valid product code format")
else:
    print("Invalid product code format")
```

Output:

```text
Valid product code format
```

If the value is `AB2048X`, it will not match because the entire string does not follow the pattern.

**Important:** `fullmatch()` checks the format only. It does not confirm that the product code exists in a real inventory system.

## 7.6 `re.finditer()`

`re.finditer()` finds all occurrences of a pattern and returns an iterator of Match Objects.

It is useful when we need both the matched text and its position.

```python
message = "Error 404 occurred. Error 503 occurred."

results = re.finditer(r"\d{3}", message)

for result in results:
    print("Code:", result.group())
    print("Position:", result.start())
```

Output:

```text
Code: 404
Position: 6
Code: 503
Position: 27
```

- `.group()` retrieves the matched text.
- `.start()` returns the starting position of the match.

## 7.7 `re.sub()`

`re.sub()` replaces text that matches a pattern.

Suppose customer messages contain phone numbers that should be hidden before sharing the messages.

```python
message = "Call 0712345678 or 0723456789 for assistance."

result = re.sub(r"\d{10}", "REDACTED", message)

print(result)
```

Output:

```text
Call REDACTED or REDACTED for assistance.
```

General syntax:

```python
re.sub(pattern, replacement, text)
```

We can also remove unwanted symbols:

```python
text = "Product: Laptop!!! Category: @Electronics"

cleaned_text = re.sub(r"[!@]", "", text)

print(cleaned_text)
```

Output:

```text
Product: Laptop Category: Electronics
```

## 7.8 `re.split()`

`re.split()` splits text wherever the specified pattern occurs.

Suppose a list of tags uses commas, semicolons, or vertical bars as separators:

```python
tags = "Python, SQL; Power BI | Excel"

result = re.split(r"[,;|]\s*", tags)

print(result)
```

Output:

```text
['Python', 'SQL', 'Power BI', 'Excel']
```

The pattern means:

- `[,;|]` matches a comma, semicolon, or vertical bar.
- `\s*` matches zero or more spaces after the separator.

## 7.9 `re.compile()`

`re.compile()` creates a reusable Regex pattern.

This is useful when we want to apply the same pattern several times.

```python
pattern = re.compile(r"ORD-\d{4}")

message_1 = "Order ORD-1024 has shipped."
message_2 = "Order ORD-5831 is being packed."

print(pattern.findall(message_1))
print(pattern.findall(message_2))
```

Output:

```text
['ORD-1024']
['ORD-5831']
```

Instead of writing the same pattern repeatedly, we create it once and reuse it.

### Student Challenge 6: Choosing Regex Functions

Given:

```python
message = "Ticket TK-204 is open, while ticket TK-518 is closed."
```

Use the appropriate Regex functions to:

1. Extract all ticket IDs.
2. Find the first ticket ID.
3. Replace all ticket IDs with `HIDDEN`.
4. Split the message at each comma.

### Reference Solution

```python
import re

message = "Ticket TK-204 is open, while ticket TK-518 is closed."

# 1. Extract all ticket IDs
all_tickets = re.findall(r"TK-\d{3}", message)

# 2. Find the first ticket ID
first_ticket = re.search(r"TK-\d{3}", message)

# 3. Replace all ticket IDs
hidden_tickets = re.sub(r"TK-\d{3}", "HIDDEN", message)

# 4. Split at each comma
parts = re.split(r",", message)

print("All tickets:", all_tickets)
print("First ticket:", first_ticket.group() if first_ticket else None)
print("Hidden tickets:", hidden_tickets)
print("Parts:", parts)
```

---

# 8. Grouping and Extracting Information

Sometimes we want to match a complete pattern but retrieve specific parts of it.

Regex provides **groups**, created using parentheses `()`.

## 8.1 Capturing Groups

Suppose product records contain a product code and its quantity:

```python
text = "PEN-104:25, BOOK-208:12, BAG-310:8"
```

We want to extract each product code and quantity separately.

```python
results = re.findall(r"([A-Z]+-\d+):(\d+)", text)

print(results)
```

Output:

```text
[('PEN-104', '25'), ('BOOK-208', '12'), ('BAG-310', '8')]
```

Break down the pattern:

- `([A-Z]+-\d+)` captures the product code.
- `:` matches the colon separator.
- `(\d+)` captures the quantity.

Because there are two capturing groups, `findall()` returns a tuple for each match.

## 8.2 Named Groups

We can give groups meaningful names. This makes extracted information easier to understand.

```python
text = "PEN-104:25, BOOK-208:12, BAG-310:8"

pattern = r"(?P<product>[A-Z]+-\d+):(?P<quantity>\d+)"

results = re.finditer(pattern, text)

for result in results:
    print("Product:", result.group("product"))
    print("Quantity:", result.group("quantity"))
```

Output:

```text
Product: PEN-104
Quantity: 25
Product: BOOK-208
Quantity: 12
Product: BAG-310
Quantity: 8
```

Named groups are useful when extracting multiple fields from structured text.

## 8.3 Non-Capturing Groups

Sometimes we need to group parts of a pattern without extracting them as a separate result.

We use `(?:...)` to create a non-capturing group.

```python
text = "Payment made by card. Payment made by cash."

result = re.findall(r"Payment made by (?:card|cash)", text)

print(result)
```

Output:

```text
['Payment made by card', 'Payment made by cash']
```

The group combines the alternatives `card` and `cash`, but does not create a separate captured group.

## 8.4 Alternation (`|`)

The vertical bar `|` means OR. It allows us to match one option or another.

```python
text = "The shipment is delayed. The order is cancelled. The payment is complete."

result = re.findall(r"delayed|cancelled", text)

print(result)
```

Output:

```text
['delayed', 'cancelled']
```

The pattern matches either `delayed` or `cancelled`.

### Student Challenge 7: Extracting Structured Information

Given:

```python
text = "PEN-104:25, BOOK-208:12, BAG-310:8"
```

Use capturing groups to:

1. Extract each product code.
2. Extract each quantity.
3. Display the product code and quantity separately.

### Reference Solution

```python
import re

text = "PEN-104:25, BOOK-208:12, BAG-310:8"

pattern = r"([A-Z]+-\d+):(\d+)"

results = re.findall(pattern, text)

for product, quantity in results:
    print("Product:", product)
    print("Quantity:", quantity)
```

---

# 9. Advanced Regex Concepts

The concepts introduced so far are enough to solve many everyday text-processing problems. Regex also provides advanced features for more specific matching tasks.

## 9.1 Greedy Matching

By default, quantifiers such as `*` and `+` are greedy. They try to match as much text as possible while still allowing the overall pattern to match.

```python
text = "Items: [Laptop] [Mouse] [Keyboard]"

result = re.findall(r"\[.*\]", text)

print(result)
```

Output:

```text
['[Laptop] [Mouse] [Keyboard]']
```

The pattern `.*` matches as much text as possible between the first `[` and the last `]`.

## 9.2 Lazy Matching

Sometimes we want the shortest possible match.

We can make a quantifier lazy by adding `?` after it.

```python
text = "Items: [Laptop] [Mouse] [Keyboard]"

result = re.findall(r"\[.*?\]", text)

print(result)
```

Output:

```text
['[Laptop]', '[Mouse]', '[Keyboard]']
```

The difference:

- `.*` is greedy and tries to match as much as possible.
- `.*?` is lazy and tries to match as little as possible.

## 9.3 Lookahead

A lookahead checks whether a pattern is followed by another pattern without including that second pattern in the returned match.

Suppose we want to extract numbers that are followed by `KES`.

```python
text = "Prices: 500 KES, 1000 USD, 250 KES"

result = re.findall(r"\d+(?= KES)", text)

print(result)
```

Output:

```text
['500', '250']
```

The pattern `(?= KES)` checks that `KES` follows the number, but does not include `KES` in the match.

## 9.4 Lookbehind

A lookbehind checks whether a pattern is preceded by another pattern.

Suppose we want to extract amounts that appear after `KES`.

```python
text = "KES 500, USD 1000, KES 250"

result = re.findall(r"(?<=KES )\d+", text)

print(result)
```

Output:

```text
['500', '250']
```

The pattern `(?<=KES )` checks that the number is preceded by `KES `.

## 9.5 Backreferences

A backreference lets us refer to text previously matched by a capturing group.

For example, we may want to find accidentally repeated words in customer feedback.

```python
text = "The delivery was was late and the package was damaged."

result = re.findall(r"\b(\w+)\s+\1\b", text)

print(result)
```

Output:

```text
['was']
```

Break down the pattern:

- `\b` identifies a word boundary.
- `(\w+)` captures a word.
- `\s+` matches one or more whitespace characters.
- `\1` matches the same text captured by the first group.
- The final `\b` marks the end of the repeated word.

Backreferences are useful when looking for repeated words or repeated text patterns.

---

# 10. Applying Regex in Data Analysis with Pandas

Regex is especially useful when working with text columns in datasets. Real-world data often contains inconsistent or unstructured text.

Let's use a small customer support dataset to extract reference numbers and clean message text.

## 10.1 Preparing the Data

```python
import pandas as pd

data = {
    "customer": ["Amina", "Brian", "Wanjiku", "David"],
    "message": [
        "My order ORD-1024 has not arrived.",
        "Please check order ORD-5831!",
        "I received order ORD-9002 yesterday.",
        "I need help with my delivery."
    ]
}

df = pd.DataFrame(data)

print(df)
```

## 10.2 Extracting Order IDs

We can use `.str.extract()` to extract the first match from each row.

```python
df["order_id"] = df["message"].str.extract(r"(ORD-\d{4})")

print(df[["customer", "order_id"]])
```

The parentheses create a capturing group. Pandas uses that group to extract the matched value.

Rows without an order ID receive a missing value (`NaN`).

## 10.3 Removing Unwanted Characters

Suppose product codes are stored in different formats:

```python
df = pd.DataFrame({
    "product_code": ["PR-204", "PR 305", "PR.406", "PR-507"]
})
```

We want to keep only letters and digits.

```python
df["clean_code"] = df["product_code"].str.replace(r"[^A-Za-z0-9]", "", regex=True)

print(df)
```

The pattern `[^A-Za-z0-9]` matches anything that is not a letter or digit. Replacing those characters with an empty string removes them.

## 10.4 Extracting Email Addresses

Consider a column containing customer messages:

```python
df = pd.DataFrame({
    "message": [
        "Send the receipt to amina@gmail.com",
        "Contact brian.k@example.org for an update",
        "No email address was provided"
    ]
})
```

We can extract common email-like patterns:

```python
pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

df["email"] = df["message"].str.extract(f"({pattern})")

print(df)
```

This pattern identifies common email address structures. It is useful for extraction, but it does not guarantee that an email address exists or is valid.

**Note:** Regex can identify common formats, but it is not a complete replacement for specialised email validation.

---

# 11. Regex Quick Reference

Use this section as a reference when writing Regex patterns.

## 11.1 Character Classes and Shortcuts

| Pattern | Meaning |
|---|---|
| `[abc]` | Matches `a`, `b`, or `c` |
| `[a-z]` | Matches lowercase English letters |
| `[A-Z]` | Matches uppercase English letters |
| `[0-9]` | Matches digits |
| `[^0-9]` | Matches anything except digits |
| `\d` | Matches a digit |
| `\D` | Matches a non-digit |
| `\w` | Matches a word character |
| `\W` | Matches a non-word character |
| `\s` | Matches whitespace |
| `\S` | Matches non-whitespace |

## 11.2 Common Metacharacters and Quantifiers

| Pattern | Meaning |
|---|---|
| `.` | Any character except a newline by default |
| `*` | Zero or more occurrences |
| `+` | One or more occurrences |
| `?` | Zero or one occurrence; also used for lazy matching |
| `{n}` | Exactly `n` occurrences |
| `{n,}` | At least `n` occurrences |
| `{n,m}` | Between `n` and `m` occurrences |
| `^` | Beginning of a string |
| `$` | End of a string |
| `\b` | Word boundary |
| `\` | Escapes a special character or introduces a special sequence |
| `|` | OR |
| `()` | Creates a capturing group |
| `(?:...)` | Creates a non-capturing group |

## 11.3 Common Regex Functions

| Function | Use |
|---|---|
| `re.findall()` | Find all matches |
| `re.search()` | Find the first match anywhere |
| `re.match()` | Check for a match at the beginning |
| `re.fullmatch()` | Check whether the entire string matches |
| `re.finditer()` | Find matches as Match Objects |
| `re.sub()` | Replace matching text |
| `re.split()` | Split text using a pattern |
| `re.compile()` | Create a reusable pattern |

---

# 12. Final Practice: Customer Support Messages

You are analysing customer support messages. Each message may contain a ticket ID, an email address, a phone number, and a status.

```python
messages = """
Ticket TK-204 | Customer: Amina | Email: amina@gmail.com | Phone: 0712345678 | Status: Open
Ticket TK-518 | Customer: Brian | Email: brian@example.org | Phone: 0723456789 | Status: Closed
Ticket TK-732 | Customer: Wanjiku | Email: wanjiku@yahoo.com | Phone: 0700123456 | Status: Open
"""
```

Using Regex, complete the following tasks:

1. Extract all ticket IDs.
2. Extract all customer names.
3. Extract all email addresses.
4. Extract all phone numbers.
5. Extract all status values.
6. Replace all phone numbers with `REDACTED`.
7. Extract each ticket ID and its status using capturing groups.
8. Find all messages with the status `Open`.

## Reference Solution

```python
import re

messages = """
Ticket TK-204 | Customer: Amina | Email: amina@gmail.com | Phone: 0712345678 | Status: Open
Ticket TK-518 | Customer: Brian | Email: brian@example.org | Phone: 0723456789 | Status: Closed
Ticket TK-732 | Customer: Wanjiku | Email: wanjiku@yahoo.com | Phone: 0700123456 | Status: Open
"""

# 1. Extract ticket IDs
ticket_ids = re.findall(r"TK-\d{3}", messages)

# 2. Extract customer names
names = re.findall(r"Customer: ([A-Za-z]+)", messages)

# 3. Extract email addresses
email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
emails = re.findall(email_pattern, messages)

# 4. Extract phone numbers
phone_numbers = re.findall(r"\d{10}", messages)

# 5. Extract status values
statuses = re.findall(r"Status: (Open|Closed)", messages)

# 6. Replace phone numbers
redacted_messages = re.sub(r"\d{10}", "REDACTED", messages)

# 7. Extract ticket IDs and statuses
ticket_statuses = re.findall(
    r"(TK-\d{3}).*?Status: (Open|Closed)",
    messages
)

# 8. Find lines with Open status
open_messages = re.findall(r"^.*Status: Open$", messages, re.MULTILINE)

print("Ticket IDs:", ticket_ids)
print("Customer names:", names)
print("Emails:", emails)
print("Phone numbers:", phone_numbers)
print("Statuses:", statuses)
print("Ticket and status:", ticket_statuses)
print("Open messages:", open_messages)

print("\nMessages with phone numbers hidden:")
print(redacted_messages)
```

## What You Should Understand

Regex is not about memorising every symbol or function. It is about learning how to describe the structure of text.

When constructing a Regex pattern, ask yourself:

1. What exactly am I trying to find?
2. What does that information look like?
3. Which characters are allowed?
4. How many times should they appear?
5. Where should the pattern appear?
6. Which Regex function is appropriate for the task?

Start with a simple pattern, test what it matches, and add complexity only when you need it.
